In [ ]:
# Dependency Installation
!pip install requests beautifulsoup4 transformers torch openai google-generativeai anthropic

In [ ]:
# Imports and API Configuration

import requests
from bs4 import BeautifulSoup
from transformers import pipeline
import openai
import google.generativeai as genai
import anthropic
import os

# --- API Key Configuration ---
# IMPORTANT: Replace the placeholder text with your actual API keys.
# It's strongly recommended to use Colab Secrets for your API keys.
# Click the key icon (🔑) on the left sidebar to add them as secrets.

# Example of using secrets (recommended):
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# CLAUDE_API_KEY = userdata.get('CLAUDE_API_KEY')

# Or paste them directly here (less secure):
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"
CLAUDE_API_KEY = "YOUR_CLAUDE_API_KEY"


# --- Model Name Constants ---
HUGGING_FACE_MODEL = "EleutherAI/gpt-neo-125M"
OPENAI_MODEL = "gpt-4o"
GEMINI_MODEL = "gemini-1.5-flash"
CLAUDE_MODEL = "claude-3-haiku-20240307"

In [ ]:
# Web Scraping Function

def scrape_website(url):
    """Scrapes text content from the paragraph tags of a given URL."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        paragraphs = soup.find_all('p')
        # Filter out empty paragraphs and join with a space
        scraped_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
        return scraped_text
    except requests.exceptions.RequestException as e:
        print(f"Error during web scraping for {url}: {e}")
        return None```

In [ ]:
# Synthetic Data Generation Functions

def generate_with_hugging_face(prompt_text, model_name=HUGGING_FACE_MODEL, max_length=200):
    """Generates synthetic text using a Hugging Face pipeline."""
    try:
        generator = pipeline('text-generation', model=model_name)
        synthetic_text = generator(prompt_text, max_length=max_length, num_return_sequences=1)
        return synthetic_text[0]['generated_text']
    except Exception as e:
        print(f"Error during Hugging Face generation: {e}")
        return None

def generate_with_openai(prompt_text, model_name=OPENAI_MODEL, max_tokens=200):
    """Generates synthetic text using the OpenAI API."""
    try:
        openai.api_key = OPENAI_API_KEY
        response = openai.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt_text}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error during OpenAI generation: {e}")
        return None

def generate_with_gemini(prompt_text, model_name=GEMINI_MODEL):
    """Generates synthetic text using the Google Gemini API."""
    try:
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt_text)
        return response.text
    except Exception as e:
        print(f"Error during Gemini generation: {e}")
        return None

def generate_with_claude(prompt_text, model_name=CLAUDE_MODEL, max_tokens=200):
    """Generates synthetic text using the Anthropic Claude API."""
    try:
        client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
        message = client.messages.create(
            model=model_name,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt_text}
            ]
        )
        return message.content[0].text
    except Exception as e:
        print(f"Error during Claude generation: {e}")
        return None

In [ ]:
# Data Saving Functions

import csv
import json
from datetime import datetime

def save_as_json(data, base_filename="synthetic_data_output"):
    """Saves the provided data list as a JSON file."""
    filename = f"{base_filename}.json"
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"Successfully saved data to {filename}")
        return filename
    except Exception as e:
        print(f"Error saving to JSON: {e}")
        return None

def save_as_csv(data, base_filename="synthetic_data_output"):
    """Saves the provided data list as a CSV file."""
    if not data:
        print("No data to save.")
        return None

    filename = f"{base_filename}.csv"
    # The fieldnames are the keys of the first dictionary in the list
    fieldnames = data[0].keys()
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
        print(f"Successfully saved data to {filename}")
        return filename
    except Exception as e:
        print(f"Error saving to CSV: {e}")
        return None

In [ ]:
# Main Execution with Output Formatting

# --- DEFINE YOUR LIST OF WEBSITES HERE ---
target_urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing"
    # Add more URLs to this list as needed
]

# 1. Choose the generation model
print("Choose a model for synthetic data generation:")
print("1: Hugging Face (EleutherAI/gpt-neo-125M)")
print("2: OpenAI (gpt-4o)")
print("3: Google Gemini (gemini-1.5-flash)")
print("4: Anthropic Claude (claude-3-haiku-20240307)")
choice = input("Enter the number of your choice: ")

all_results = []
model_map = {'1': 'Hugging Face', '2': 'OpenAI', '3': 'Gemini', '4': 'Claude'}
chosen_model_name = model_map.get(choice, 'Unknown')

# 2. Loop through each URL
for url in target_urls:
    print(f"\n{'='*50}\nProcessing URL: {url}\n{'='*50}")

    scraped_content = scrape_website(url)

    if scraped_content:
        prompt = scraped_content[:1000]
        print("\n--- Scraped Content (Prompt) ---")
        print(prompt + "\n")

        synthetic_data = None
        if choice == '1':
            print("Generating with Hugging Face...")
            synthetic_data = generate_with_hugging_face(prompt)
        elif choice == '2':
            print("Generating with OpenAI...")
            synthetic_data = generate_with_openai(prompt)
        elif choice == '3':
            print("Generating with Gemini...")
            synthetic_data = generate_with_gemini(prompt)
        elif choice == '4':
            print("Generating with Claude...")
            synthetic_data = generate_with_claude(prompt)
        else:
            print("Invalid model choice provided. Skipping generation.")
            continue

        if synthetic_data:
            print("--- Generation Complete ---")
            # Append a structured dictionary to our results list
            all_results.append({
                "source_url": url,
                "generation_model": chosen_model_name,
                "timestamp_utc": datetime.utcnow().isoformat(),
                "prompt_text": prompt,
                "generated_synthetic_data": synthetic_data
            })
        else:
            print("--- Generation Failed ---")
    else:
        print(f"Could not scrape content from {url}. Skipping.")

# 3. Save the collected results to a file
if all_results:
    print(f"\n\n{'='*60}")
    print("      DATA GENERATION COMPLETE. CHOOSE OUTPUT FORMAT")
    print(f"{'='*60}\n")
    print("1: Save as JSON")
    print("2: Save as CSV")
    print("3: Save as both JSON and CSV")
    print("Any other key: Skip saving and just print to console")
    save_choice = input("Enter your choice: ")

    if save_choice == '1':
        save_as_json(all_results)
    elif save_choice == '2':
        save_as_csv(all_results)
    elif save_choice == '3':
        save_as_json(all_results)
        save_as_csv(all_results)
    else:
        print("\n--- SKIPPING FILE SAVE. DISPLAYING RESULTS IN CONSOLE ---\n")
        print(json.dumps(all_results, indent=2))

else:
    print("\nNo synthetic data was generated. Nothing to save.")